# 01 — Exploration des données

**Objectif :** comprendre les données avant de modéliser. On vérifie :
1. la répartition tricheurs / honnêtes (déséquilibre des classes) ;
2. les variables qui distinguent les tricheurs ;
3. les **pièges d'équité** identifiés dans le cahier des charges (manette, accessibilité) ;
4. pourquoi la règle actuelle « trier par signalements » est insuffisante ;
5. la qualité des données (valeurs manquantes, contrat).

Les données sont synthétiques et générées par `src/data/generate.py` (graine fixe : reproductibles).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Permet d'importer le code du dossier src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.viz import BLUE, HONEST_CHEATER_COLORS, HONEST_CHEATER_LABELS, ORANGE, apply_style

apply_style()
pd.set_option("display.precision", 3)

In [ ]:
from src.config import NUMERIC_FEATURES, TARGET
from src.data import generate_dataset, validate_dataset

df = generate_dataset(n_matches=50_000, seed=42)
validate_dataset(df)  # lève une erreur claire si le contrat de données n'est pas respecté
print(f"{len(df):,} fiches (joueur, match) — {df['player_id'].nunique():,} joueurs")
df.head()

## 1. Déséquilibre des classes

In [ ]:
counts = df[TARGET].value_counts().rename(index=HONEST_CHEATER_LABELS)
print(counts.to_string())
print(f"\nPart de tricheurs : {df[TARGET].mean():.1%}")

Environ **3 % de tricheurs** : un modèle qui répondrait toujours « honnête » aurait 97 % de bonnes réponses tout en étant inutile.
➡️ On n'utilisera **pas l'accuracy**, mais la PR-AUC, la précision sur les cas les plus suspects et le taux de faux positifs (voir notebook 02).

## 2. Les signaux de triche

In [ ]:
signals = [
    ("headshot_rate", "Taux de tirs à la tête"),
    ("avg_reaction_ms", "Temps de réaction moyen (ms)"),
    ("kills_through_smoke_ratio", "Éliminations à travers la fumée"),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (column, title) in zip(axes, signals):
    for label in (0, 1):
        values = df.loc[df[TARGET] == label, column].dropna()
        ax.hist(
            values,
            bins=40,
            density=True,
            alpha=0.55,
            color=HONEST_CHEATER_COLORS[label],
            label=HONEST_CHEATER_LABELS[label],
        )
    ax.set_title(title)
    ax.set_ylabel("Densité")
axes[0].legend()
fig.suptitle("Distribution des signaux : honnêtes vs tricheurs", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby(TARGET)[NUMERIC_FEATURES].median().rename(index=HONEST_CHEATER_LABELS).T

**Lecture :** les tricheurs visent plus souvent la tête, réagissent plus vite et éliminent plus à travers la fumée.
Mais les distributions **se chevauchent fortement** : de nombreux tricheurs « discrets » ressemblent à des honnêtes, et des honnêtes très doués ressemblent à des tricheurs. Aucune variable seule ne suffit → il faut un modèle qui combine les signaux.

## 3. Piège d'équité n°1 : l'assistance à la visée de la manette

In [ ]:
honest = df[df[TARGET] == 0]
by_device = honest.groupby("input_device")[["accuracy", "headshot_rate"]].mean()
print(by_device.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
for color, (device, rows) in zip([BLUE, ORANGE], honest.groupby("input_device")):
    ax.hist(rows["accuracy"], bins=40, density=True, alpha=0.55, color=color, label=device)
ax.set_title("Précision des joueurs HONNÊTES selon le périphérique")
ax.set_xlabel("Précision des tirs")
ax.set_ylabel("Densité")
ax.legend()
plt.show()

Les joueurs **honnêtes** à la manette sont **plus précis** (assistance à la visée intégrée au jeu).
➡️ Risque : un modèle naïf pourrait les soupçonner à tort. C'est pourquoi `input_device` est fourni au modèle (pour qu'il « sache » qu'une forte précision est normale à la manette) et pourquoi le taux de faux positifs est audité par périphérique (notebook 03).

## 4. Piège d'équité n°2 : l'option d'accessibilité

In [ ]:
honest.groupby("accessibility_mode")["avg_reaction_ms"].describe()[["mean", "std"]].round(1)

Les joueurs qui utilisent l'option d'accessibilité ont des temps de réaction **plus longs et plus variables**.
Cette variable est un **réglage technique**, **jamais utilisée par le modèle** : elle sert uniquement à vérifier, de façon agrégée, que ces joueurs ne sont pas pénalisés (cahier des charges, section 8).

## 5. Pourquoi « trier par signalements » ne suffit pas

In [ ]:
table = df.groupby("reports_received")[TARGET].agg(["count", "mean"]).head(8)
table.columns = ["Nombre de fiches", "Part de tricheurs"]
print(table.round(3).to_string())

quartiles = ["Q1", "Q2", "Q3", "Q4 (meilleurs)"]
talent = honest.assign(niveau=pd.qcut(honest["kd_ratio"], 4, labels=quartiles))
print("\nSignalements moyens des joueurs HONNÊTES selon leur ratio éliminations/morts :")
print(talent.groupby("niveau", observed=True)["reports_received"].mean().round(2).to_string())

Les signalements augmentent bien avec la triche, mais **les meilleurs joueurs honnêtes sont aussi beaucoup signalés** (adversaires frustrés).
➡️ La règle actuelle confond talent et triche : c'est le problème métier que Sentinel doit résoudre.

## 6. Qualité des données

In [ ]:
missing = df.isna().mean().loc[lambda s: s > 0]
print("Part de valeurs manquantes :")
print(missing.map("{:.1%}".format).to_string())

Seul le temps de réaction comporte des valeurs manquantes (~2 %, perte de télémétrie simulée). Elles sont **autorisées par le contrat** et remplacées par la médiane dans le pipeline du modèle (`src/features.py`).

## Conclusions
- Classes très déséquilibrées → métriques adaptées (PR-AUC, précision sur le top 1 %, faux positifs).
- Signaux réels mais chevauchants → modèle combinant plusieurs variables.
- Deux pièges d'équité confirmés → audit obligatoire (notebook 03).
- La règle des signalements confond talent et triche → référence à battre (notebook 02).